## Inicialización

In [3]:
# Importar librerias requeridas
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Flatten
from tensorflow.keras.optimizers import Adam

## Carga los datos

El conjunto de datos se almacena en la carpeta `/datasets/faces/` 
- La carpeta `final_files` con 7600 fotos 
- El archivo `labels.csv` con etiquetas, con dos columnas: `file_name` y `real_age` 
Dado que el número de archivos de imágenes es bastante elevado, se recomienda evitar leerlos todos a la vez, ya que esto consumiría muchos recursos computacionales. Te recomendamos crear un generador con ImageDataGenerator. Este método se explicó en el capítulo 3, lección 7 de este curso.

El archivo de etiqueta se puede cargar como un archivo CSV habitual.

### Nota
El cuaderno lo monté a través de Google Colab para poder usar la GPU. Bajo este entorno se realiza lo siguiente:
1. Se suben las carpetas de 'final_files' y 'labels' en Google Drive en la ruta Mi unidad/TripleTen_Projects/faces/.
2. Enlazo Drive a Google Colab para copiar y descomprimir los archivos en Google Colab para optimizar el gasto computacional.

In [ ]:
# Enlazar google.colab
from google.colab import drive
drive.mount('/content/drive')

# Carpeta de Google Drive donde guardé los archivos para este proyecto
drive_path = '/content/drive/MyDrive/TripleTen_Projects/faces/'

# Copia de la carpeta en Colab 
path = '/content/faces/'
os.makedirs(path + 'final_files', exist_ok=True)
shutil.copy(drive_path + 'labels.csv', path + 'labels.csv')

# Descomprimir carpeta final_files
if len(os.listdir(path + 'final_files')) == 0:
    with zipfile.ZipFile(drive_path + 'final_files.zip') as z:
        z.extractall(path + 'final_files')

print('Imágenes disponibles: ', len(os.listdir(path + 'final_files')))





In [ ]:
# Revisión archivo 'labels'
labels = pd.read_csv(path + 'labels.csv')
print('Forma del conjunto de datos: ', labels.shape)
labels.head()

### Observaciones
De acuerdo con el archivo 'final_files', se tienen características para 7591 personas.
De acuerdo con el archivo 'labels', se tienen 7591 registros de edad de personas que deben encajar con las imágenes del archivo anterior.

## EDA

In [ ]:
# Estructura de los datos
print(labels.info())
print()

# Valores ausentes o duplicados
print('Valores ausentes: ', labels.isna().sum().to_dict())
print('Archivos duplicados: ', labels['file_name'].duplicated().sum())
print()

# ¿todas las etiquetas tienen su foto, y viceversa?
files_in_folder = set(os.listdir(path + 'final_files'))
files_in_labels = set(labels['file_name'])
print('Etiquetas sin foto:', len(files_in_labels - files_in_folder))
print('Fotos sin etiqueta:', len(files_in_folder - files_in_labels))

labels['real_age'].describe().round(2)




In [ ]:
# Distribución de la edad (Comportamiento gráfico)
fig, ax = plt.subplots(figsize=(12,4))
labels['real_age'].plot.hist(bins=50, ax=ax)
ax.set_xlabel('Edad real (años)')
ax.set_ylabel('Número de fotos disponibles')
ax.set_title('Distribución de la edad en el juego de datos')
plt.tight_layout()
plt.show()

# Establecer rangos de edad y determinar número de fotos en cada rango:
bins = [0, 12, 17, 20, 30, 40, 50, 60, 100]
age_groups = pd.cut(labels['real_age'], bins=bins).value_counts().sort_index()

age_table = pd.DataFrame({'Fotos': age_groups, 'Porcentaje': (age_groups / len(labels) * 100).round(1)})
print(age_table)

# Determinación de los grupos de menores de edad
print(f"Menores de 18 años: {(labels['real_age'] < 18).sum()} ({(labels['real_age'] < 18).mean():.1%})")
print(f"Menores de 21 años: {(labels['real_age'] < 21).sum()} ({(labels['real_age'] < 21).mean():.1%})")
print()


In [ ]:
# Muestra de fotografías
sample = labels.sample(15, random_state=792604)

fig, axes = plt.subplots(3, 5, figsize=(15,9))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open(path + 'final_files/' + row['file_name']))
    ax.set_title(f"'{row['real_age']} años")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Revisión del támaño de las imagenes
widths, heights, modes = [], [], []
for file_name in labels.sample(300, random_state=792604)['file_name']:
    with Image.open(path + 'final_files/' + file_name) as img:
        widths.append(img.size[0])
        heights.append(img.size[1])
        modes.append(img.mode)

sizes = pd.DataFrame({'ancho': widths, 'alto': heights})
print('Modos de color:', pd.Series(modes).value_counts().to_dict())
print('Relación alto/ancho (mín, máx):', round((sizes['alto'] / sizes['ancho']).min(), 2), round((sizes['alto'] / sizes['ancho']).max(), 2))
sizes.describe().round(1)

### Conclusiones
De la estructura de datos del archivo 'labels':
- El juego de datos tiene 2 columnas, la columna 'file_name' es tipo object, la cual corresponde al título de la imagen o foto. La segunda columna 'real_age' corresponde a la edad de la persona en la foto y es de tipo int64.
- No se tienen valores ausentes ni duplicados en el juego de datos.
- Cada foto de 'final_files' corresponde a un archivo de 'labels'.
- De acuerdo a los datos de 'real_age', la media de los datos de edad corresponde a 31 años, con valor mínimo de 1 y máximo de 100. La desviación estándar es de 17 años aproximadamente, y los datos del percentil 25 al 75 son de un rango de edad de 20 a 41. Este es un valor útil para el proyecto, porque datos de edad mayores e incluso menores saldrán a simple vista por el vendedor del supermercado, mientras el grueso del rango de edad podrá ser manejado por el modelo de visión artificial.

Luego se realiza un estudio gráfico para corroborar la última conclusión descrita, mediante un histograma. Se encuentra:
- Efectivamente, el grueso de los datos de edad disponibles en el conjunto de imagenes va en un rango de 15 a 50 años de edad.
- Se establecen grupos de edad para determinar niños (1-12 años), adolescentes (13-17 años), franja de mayoría de edad (18-20 años), y franjas de mayores de edad entre 21-30 años, de 31-40 años, 41-50 años, 51-60 años, 61-100 años. cerca del 50% de los datos disponibles se tiene en el rango de 21-40 años.
- Se revisa la cantidad de fotos de personas menores de 18 años y menores de 21 años. El ejercicio no define cúal es la mayoria de edad, entonces se toman estas 2 franjas. Aproximadamente el 20% de las fotos corresponden a menores de 18 años, y sube a 27% si son menores de 21 años. Esto permitirá obtener un modelo robusto en las predicciones ya que cuenta con datos suficientes para la evaluación.

Se monta un muestreo aleatorio del conjunto de datos de 15 fotos, con su respectiva edad para corroborar que no se vaya a presentar discrepancias. En términos generales, las fotos de las personas guardan relación con la edad reportada. **No se encuentran discrepancias evidentes en la muestra y podría inferirse que el juego de archivos usado para montar el modelo guarda confiabilidad**.

Finalmente, se toma un muestreo aleatorio de 300 fotos para determinar su tamaño, encontrando que el tamaño medio de esta muestra es de 457x454 píxeles, con fotos que van de 47x47 a 3381x3381 píxeles. Esto indica que es **necesario realizar un reescalado de los archivos al momento de realizar el modelamiento**.

## Modelado

Define las funciones necesarias para entrenar tu modelo en la plataforma GPU y crea un solo script que las contenga todas junto con la sección de inicialización.

Para facilitar esta tarea, puedes definirlas en este notebook y ejecutar un código listo en la siguiente sección para componer automáticamente el script.

In [ ]:
def load_train(path):
    
    """
    Carga la parte de entrenamiento del conjunto de datos desde la ruta.
    """
    labels = pd.read_csv(path + 'labels.csv')

    train_datagen = ImageDataGenerator(
        validation_split=0.25,
        rescale=1./255,
        horizontal_flip=True
    )
    
    train_gen_flow = train_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='training',
        seed=792604
    )

    return train_gen_flow

In [ ]:
def load_test(path):
    
    """
    Carga la parte de validación/prueba del conjunto de datos desde la ruta
    """
    labels = pd.read_csv(path + 'labels.csv')

    test_datagen = ImageDataGenerator(
        validation_split=0.25,
        rescale=1./255
    )

    test_gen_flow = test_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='validation',
        shuffle=False,
        seed=792604
    )
    
    return test_gen_flow

In [ ]:
def create_model(input_shape):
    
    """
    Define el modelo
    """
    
    backbone = ResNet50(input_shape=input_shape,
                       weights='imagenet',
                       include_top=False)

    #Modelo
    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(1, activation='relu'))

    optimizer = Adam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    return model

In [ ]:
def train_model(model, train_data, test_data, batch_size=None, epochs=20,
                steps_per_epoch=None, validation_steps=None):

    """
    Entrena el modelo dados los parámetros
    """
    
    if steps_per_epoch is None:
        steps_per_epoch = len(train_data)
    if validation_steps is None:
        validation_steps = len(test_data)

    model.fit(train_data,
             validation_data=test_data,
             epochs=epochs,
             steps_per_epoch=steps_per_epoch,
             validation_steps=validation_steps,
             verbose=2)

    return model

### Observaciones
En la creación de las funciones de modelado se toma lo siguiente:
- La carga de datos se toma con **flow_from_dataframe** y no con *flow_from_directory*, esto básicamente porque se parte de un archivo csv y no de una carpeta con archivos por clase como podría pasar con los ejercicios de clasificación de frutas.
- Se utiliza **class_mode='raw'** y no *class_mode='sparse'* o *'categorical'* debido a que se tiene un problema de regresión con una variable continua (la edad).
- Como se instruyó en el sprint, se parte el archivo en 75% para entrenamiento y 25% para validación.
- Se ajusta el tamaño de imagen a 224x224 que es la forma usual de manejo con ResNet50. De igual forma, se normalizan las imágenes para llevar los datos de pixeles a un rango [0, 1].
- El aumento de datos solo se realiza en el conjunto de entrenamiento, y solo se hace un "volteo" horizontal. Dado que son fotos de rostros, no se justifica un volteo vertical. Tampoco se busca alterar las imagenes ya que será una determinación de edad.
- Se parte de un modelo ResNet50 ya que es un modelo robusto y dato el juego de datos para entrenamiento, es mejor usar un modelo con entrenamiento previo.
- Este modelo se deja con capas entrenables que permitan un ajuste fino.
- Como la edad no puede ser negativa, se toma una activación *'relu'*.
- De acuerdo a las instrucciones del proyecto, se maneja error absoluto medio y error cuadrático medio como métricas.
- Se toma un optimizador **Adam** con una tasa de aprendizaje baja para evitar destruir los pesos preentrenados durante el ajuste fino.


## Prepara el script para ejecutarlo en la plataforma GPU

Una vez que hayas definido las funciones necesarias, puedes redactar un script para la plataforma GPU, descargarlo a través del menú "File|Open..." (Archivo|Abrir) y cargarlo más tarde para ejecutarlo en la plataforma GPU.

Nota: el script debe incluir también la sección de inicialización. A continuación se muestra un ejemplo.

In [ ]:
# prepara un script para ejecutarlo en la plataforma GPU

init_str = """
import pandas as pd

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
"""

import inspect

with open('run_model_on_gpu.py', 'w') as f:
    
    f.write(init_str)
    f.write('\n\n')
        
    for fn_name in [load_train, load_test, create_model, train_model]:
        
        src = inspect.getsource(fn_name)
        f.write(src)
        f.write('\n\n')

In [ ]:
# Revisión de que Colab este corriendo con GPU
print('GPU disponible: ', tf.config.list_physical_devices('GPU'))

In [ ]:
# Ejecución del modelado
train_gen_flow = load_train(path)
test_gen_flow = load_test(path)

model = create_model(input_shape=(224, 224, 3))
model = train_model(model, train_gen_flow, test_gen_flow, epochs=20)

In [ ]:
# Cálculo del error absoluto medio en el conjunto de validación
test_loss, test_mae = model.evaluate(test_gen_flow, verbose=0)
print(f'EAM (MAE) en el conjunto de prueba: {test_mae:.2f} años')

### El resultado

El ejercicio ejecutado en Colab entrega estos resultados:
'''
Found 5694 validated image filenames.
Found 1897 validated image filenames.

Epoch 1/20
178/178 - 183s - 1s/step - loss: 244.9211 - mae: 11.1523 - val_loss: 714.3138 - val_mae: 21.7877

Epoch 2/20
178/178 - 72s - 403ms/step - loss: 79.6969 - mae: 6.6959 - val_loss: 551.4149 - val_mae: 18.2820

Epoch 3/20
178/178 - 75s - 420ms/step - loss: 53.8412 - mae: 5.5461 - val_loss: 212.6302 - val_mae: 11.2595

Epoch 4/20
178/178 - 77s - 431ms/step - loss: 43.0463 - mae: 4.9919 - val_loss: 120.7305 - val_mae: 8.4529

Epoch 5/20
178/178 - 75s - 421ms/step - loss: 32.0650 - mae: 4.2814 - val_loss: 121.6405 - val_mae: 8.6527

Epoch 6/20
178/178 - 82s - 462ms/step - loss: 25.4153 - mae: 3.8528 - val_loss: 66.5321 - val_mae: 5.9851

Epoch 7/20
178/178 - 74s - 415ms/step - loss: 21.5997 - mae: 3.5693 - val_loss: 60.4948 - val_mae: 5.8163

Epoch 8/20
178/178 - 83s - 466ms/step - loss: 17.2878 - mae: 3.1859 - val_loss: 79.7768 - val_mae: 6.6427

Epoch 9/20
178/178 - 74s - 417ms/step - loss: 17.4267 - mae: 3.1973 - val_loss: 91.5523 - val_mae: 7.4151

Epoch 10/20
178/178 - 75s - 420ms/step - loss: 16.3190 - mae: 3.1120 - val_loss: 66.5459 - val_mae: 6.2138

Epoch 11/20
178/178 - 75s - 420ms/step - loss: 16.2237 - mae: 3.1072 - val_loss: 63.1314 - val_mae: 5.8812

Epoch 12/20
178/178 - 74s - 416ms/step - loss: 14.5339 - mae: 2.9338 - val_loss: 68.2763 - val_mae: 6.1966

Epoch 13/20
178/178 - 75s - 423ms/step - loss: 14.3178 - mae: 2.8819 - val_loss: 75.7371 - val_mae: 6.4369

Epoch 14/20
178/178 - 75s - 420ms/step - loss: 12.4049 - mae: 2.6860 - val_loss: 59.6639 - val_mae: 5.7330

Epoch 15/20
178/178 - 75s - 419ms/step - loss: 11.5661 - mae: 2.6032 - val_loss: 58.9938 - val_mae: 5.6704

Epoch 16/20
178/178 - 75s - 422ms/step - loss: 11.3847 - mae: 2.5541 - val_loss: 62.4417 - val_mae: 5.9704

Epoch 17/20
178/178 - 75s - 422ms/step - loss: 11.7278 - mae: 2.6292 - val_loss: 67.5467 - val_mae: 5.9725

Epoch 18/20
178/178 - 74s - 417ms/step - loss: 10.8572 - mae: 2.5129 - val_loss: 63.2474 - val_mae: 5.8470

Epoch 19/20
178/178 - 75s - 421ms/step - loss: 9.7291 - mae: 2.3856 - val_loss: 68.4298 - val_mae: 6.2819

Epoch 20/20
178/178 - 75s - 420ms/step - loss: 9.4102 - mae: 2.3551 - val_loss: 58.5054 - val_mae: 5.6657
'''

**Error absoluto medio:**
En el conjunto de prueba, el error absoluto medio es de 5.67 años.

### Observaciones
- Una vez se corren las 20 épocas del modelo se obtiene un *error absoluto medio* para el conjunto de validación de 5.67 años. Teniendo en cuenta que la meta era menor a 8 años, el modelo cumple con el deseo del cliente.
- Se debe mencionar que si bien el *error absoluto medio* para el conjunto de entrenamiento fue disminuyendo constantemente con el pasar de las épocas, el error para el conjunto de validación se fue "normalizando" después de la época 7.
- La diferencia de errores en los conjuntos de entrenamiento y validación al final es de 3.3 años. Esto puede ser un leve síntome de sobreajuste de los datos.
- El modelo empieza a arrojar valores de error aceptables después de la quinta época.

## Conclusiones

La cadena de supermercados Good Seed desea implementar un sistema de detección de edad de los clientes que compran bebidas alcohólicas, aprovechando el sistema de cámaras que disponen en sus puntos de pago. Para ello se utiliza una base de datos de 7591 fotos de rostros de personas de edades que oscilan entre 1 y 100 años. Para ello:
- Los datos son revisados y acondicionados previamente para asegurar que el modelo implementado sea lo más robusto posible.
- Se implementa un modelo ResNet50 con ajuste fino y una neurona de salida. El modelo entrenado se prueba con un conjunto de validación, alcanzando un **Error Absoluto Medio de 5.67 años** sobre un valor máximo esperado de 8 años. Este valor se obtiene con un conjunto de datos que se vio durante el entrenamiento y no sobre un conjunto independiente.
- Durante el entrenamiento, el error en el conjunto de validación decrece de forma rápida durante las primeras 7 épocas, luego muestra una estabilización entre 5 y 7 años sin una mejora significativa. Para el conjunto de entrenamiento esta reducción si continua decreciendo con el pasar de las épocas hasta llegar a un valor de 2.36.
- Aunque la meta del modelo era obtener un error de 8 años y se obtiene un valor de 5.7 años, esto en la práctica significa que la cámara puede confundir a una persona de 17 años con una de 23. Este modelo sirve como apoyo para el personal del supermercado pero no puede ser el criterio único para aprobar la venta.

# Lista de control

- [x]  El Notebook estaba abierto 
- [x]  El código no tiene errores
- [x]  Las celdas con el código han sido colocadas en el orden de ejecución
- [x]  Se realizó el análisis exploratorio de datos 
- [x]  Los resultados del análisis exploratorio de datos se presentan en el notebook final 
- [x]  El valor EAM del modelo no es superior a 8 
- [x]  El código de entrenamiento del modelo se copió en el notebook final 
- [x]  El resultado de entrenamiento del modelo se copió en el notebook final 
- [x] Los hallazgos se proporcionaron con base en los resultados del entrenamiento del modelo